# Add-ons
Exploratory code, not yet in the package.

In [1]:
import dataclasses
import glob
import os
from dataclasses import dataclass

import numpy as np
from joblib import Parallel, delayed

from neuropy import core
from neuropy.analyses.ms_connectivity import open_project
from neuropy.analyses.neurons_dataset import Key, NeuronsDataset
from neuropy.analyses.placefields import Pf1D, Pf1Dsplit
from neuropy.utils.data_storage_util import atomic_write_json

/Users/selinl/miniforge3/envs/NeuroPy2/lib/python3.11/site-packages/xarray/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Error importing CuPy


## Place-field curves
Unthresholded 1D tuning curves per session, maze block and run direction; recipe from `Nat_sd_placefields_remaze.ipynb`.

In [2]:
BLOCKS = {'maze': ('maze', 'maze_run'), 'remaze': ('remaze', 'remaze_run')}   # block -> (position, run epochs)
DIRECTIONS = ('up', 'down')
PF_KW = dict(frate_thresh=0, grid_bin=5, speed_thresh=3, sigma=0)   # Pf1D defaults, minus the rate cut
SIGMA_POS = 0.1
T_SPLIT = 60   # s


def select_maze_epochs_for_pf_estimate(sess, block="maze", direction="up", speed_thresh=10):
    """Nat_sd_placefields_remaze practice: one direction's running bouts (.running.npy) where raw speed > speed_thresh."""
    position, run = (sess.maze, sess.maze_run) if block == "maze" else (sess.remaze, sess.remaze_run)
    run_bool = position.speed > speed_thresh
    run_epochs = core.Epoch.from_boolean_array(run_bool, t=position.time)
    return run[direction].intersection(run_epochs, res=1 / sess.maze.sampling_rate)


@dataclass
class PlacefieldCurves:
    """One (session, block, direction): rate per neuron per position bin, and the same from alternating T_SPLIT blocks."""
    neuron_ids: np.ndarray
    coords: np.ndarray          # left edge of each position bin
    tuning_curves: np.ndarray   # (neuron, bin), Hz
    split_curves: np.ndarray    # (2, neuron, bin): even and odd blocks

    @classmethod
    def compute(cls, neurons: core.Neurons, position: core.Position, run_thresh: core.Epoch) -> 'PlacefieldCurves':
        pf = Pf1D(neurons, position=position, epochs=run_thresh, sigma_pos=SIGMA_POS, **PF_KW)
        split = Pf1Dsplit(neurons, position=position, epochs=run_thresh, t_interval_split=T_SPLIT, **PF_KW)
        return cls(np.asarray(pf.neuron_ids), pf.x_coords(), pf.tuning_curves,
                   np.stack([split.pf1.tuning_curves, split.pf2.tuning_curves]))


FIELDS = [f.name for f in dataclasses.fields(PlacefieldCurves)]


class PlacefieldCurveSet:
    """Curves keyed (session, block, direction); one ``<session>.npz`` per session beside ``params.json``."""

    def __init__(self, root: str, curves: dict = None):
        self.root = root
        self.curves: dict = curves or {}

    @classmethod
    def compute(cls, nd: NeuronsDataset, root: str) -> 'PlacefieldCurveSet':
        """Every block and direction a session has linear position and run epochs for."""
        curves = {}
        for session in nd._sessions:
            name = session.session_name
            neurons = nd.neurons_for(Key(session=name))
            for block, (pos_attr, run_attr) in BLOCKS.items():
                position = getattr(session, pos_attr, None)
                if position is None or getattr(session, run_attr, None) is None:   # sessions without this block, or not yet linearised
                    print(f"[placefields] {name} {block}: no linear position or run epochs, skipped", flush=True)
                    continue
                for direction in DIRECTIONS:
                    print(f"[placefields] {name} {block} {direction}", flush=True)
                    curves[(name, block, direction)] = PlacefieldCurves.compute(
                        neurons, position,
                        select_maze_epochs_for_pf_estimate(session, block, direction, PF_KW['speed_thresh']))
        return cls(root, curves)

    def save(self) -> None:
        os.makedirs(self.root, exist_ok=True)
        atomic_write_json(os.path.join(self.root, 'params.json'),
                          {'pf1d': PF_KW, 'sigma_pos': SIGMA_POS, 't_interval_split': T_SPLIT})
        for session in sorted({s for s, _, _ in self.curves}):
            np.savez(os.path.join(self.root, f"{session}.npz"),
                     **{f"{b}.{d}.{field}": getattr(c, field)
                        for (s, b, d), c in self.curves.items() if s == session for field in FIELDS})

    @classmethod
    def load(cls, root: str) -> 'PlacefieldCurveSet':
        curves = {}
        for path in sorted(glob.glob(os.path.join(root, '*.npz'))):
            session = os.path.basename(path)[:-len('.npz')]
            with np.load(path) as z:
                for b, d in {tuple(k.split('.')[:2]) for k in z.files}:
                    curves[(session, b, d)] = PlacefieldCurves(*(z[f"{b}.{d}.{field}"] for field in FIELDS))
        return cls(root, curves)

In [3]:
_, cd, _ = open_project('test2')
pf_root = os.path.join(cd.save_path, 'placefields')
# PlacefieldCurveSet.compute(cd.nd, pf_root).save()   # ~7 min
pf_curves = PlacefieldCurveSet.load(pf_root)
sorted(pf_curves.curves)[:5]

[CCGPointer] loaded 64 pointers ← /Users/selinl/Documents/ms_synchrony/NeuroPy/data/project_test2/ccg/pointers/test2_lowres_*.hkl


[('RatK_Day1', 'maze', 'down'),
 ('RatK_Day1', 'maze', 'up'),
 ('RatK_Day2', 'maze', 'down'),
 ('RatK_Day2', 'maze', 'up'),
 ('RatN_Day1', 'maze', 'down')]

In [5]:
pf_curves.curves

{('RatK_Day1',
  'maze',
  'down'): PlacefieldCurves(neuron_ids=array([ 0,  4,  6,  7,  8, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22,
        23, 27, 28, 30, 31, 32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44,
        45, 46, 47, 48, 49, 50, 51, 54, 55, 56]), coords=array([1.33080703e-01, 5.13308070e+00, 1.01330807e+01, 1.51330807e+01,
        2.01330807e+01, 2.51330807e+01, 3.01330807e+01, 3.51330807e+01,
        4.01330807e+01, 4.51330807e+01, 5.01330807e+01, 5.51330807e+01,
        6.01330807e+01, 6.51330807e+01, 7.01330807e+01, 7.51330807e+01,
        8.01330807e+01, 8.51330807e+01, 9.01330807e+01, 9.51330807e+01,
        1.00133081e+02, 1.05133081e+02, 1.10133081e+02, 1.15133081e+02,
        1.20133081e+02, 1.25133081e+02, 1.30133081e+02, 1.35133081e+02,
        1.40133081e+02, 1.45133081e+02, 1.50133081e+02, 1.55133081e+02,
        1.60133081e+02, 1.65133081e+02, 1.70133081e+02, 1.75133081e+02,
        1.80133081e+02, 1.85133081e+02, 1.90133081e+02, 1.95133081e+02,
        2

## Best channels
From `sleep_loss_hippocampal_replay/.../sd_best_channels.ipynb`; helpers in `signal_process_utils.py` and `dip.py` here. Needs the full multi-channel `.eeg`.

In [ ]:
from signal_process_utils import get_sw_theta_bimodal_params   # copied from sleep_loss DataPaths, with dip.py


def pick_best_channels(sess):
    """Most bimodal connected channel for slow wave and for theta, over sampled AW/QW/NREM/REM epochs."""
    aw = sess.brainstates_fine["AW"].duration_slice(10, 500)[:15]
    qw = sess.brainstates_fine["QW"].duration_slice(10, 500)[:15]
    nrem = sess.brainstates_fine["NREM"].duration_slice(250, 500)[:5]
    rem = sess.brainstates_fine["REM"].duration_slice(100, 200)[:3]
    states = aw + qw + nrem + rem

    good_channels = np.concatenate(sess.probegroup.get_connected_channels()).astype("int")
    lfps = sess.eegfile.get_frames_within_epochs(states, good_channels)
    bimodal_params = np.array(Parallel(n_jobs=8)(delayed(get_sw_theta_bimodal_params)(_) for _ in lfps))
    return {"slow_wave": good_channels[np.argmax(bimodal_params[:, 0])],
            "theta": good_channels[np.argmax(bimodal_params[:, 2])]}

In [ ]:
# sess = next(s for s in cd.nd._sessions if s.session_name == 'RatR_Day2SD')
# pick_best_channels(sess)